# 2. Random Variables

**Statistical Foundations for Data Science — Notebook 2 of 8**

In Notebook 1 our outcomes were things like `"heads"` or `(3, 4)`. Useful, but you cannot
average a coin flip. A **random variable** fixes that: it attaches a *number* to every
outcome, and numbers can be added, averaged, squared and plotted.

Every column in your dataset is a realisation of a random variable. Every model output is
a random variable. This notebook is where statistics becomes computational.

### What you will learn

1. What a random variable really is (a function, not a variable)
2. Discrete vs. continuous random variables
3. **PMF**, **PDF** and **CDF** — and when to use each
4. **Expectation** $E[X]$: the long-run average
5. **Variance** and **standard deviation**: how spread out things are
6. Linearity of expectation, and why variance is *not* linear
7. Standardisation (z-scores), skewness and kurtosis
8. Joint, marginal and conditional distributions
9. **LOTUS** — expectation of a function of a random variable

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=7)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

---
## 2.1 What is a random variable?

> A **random variable** $X$ is a function that maps every outcome in the sample space to
> a real number: $X : S \rightarrow \mathbb{R}$.

It is neither random nor a variable — it is a deterministic rule. The randomness lives in
*which outcome occurs*; $X$ just translates that outcome into a number.

**Example.** Flip a coin three times. $S$ has 8 outcomes (`HHH`, `HHT`, …). Define
$X = $ *number of heads*. Then $X(\texttt{HHT}) = 2$, $X(\texttt{TTT}) = 0$, and so on.

Notation convention (worth internalising early):
- Capital $X$ = the random variable
- lower-case $x$ = a particular value it took
- $P(X = x)$ = the probability it takes that value

In [ ]:
from itertools import product

S = ["".join(o) for o in product("HT", repeat=3)]
X = {outcome: outcome.count("H") for outcome in S}     # the random variable as a mapping

pd.DataFrame({"outcome": S,
              "X = number of heads": [X[o] for o in S],
              "probability of outcome": [1/8] * 8})

### Discrete vs. continuous

| | **Discrete** | **Continuous** |
|---|---|---|
| Values | Countable (0, 1, 2, …) | Any value in an interval |
| Described by | **PMF** $p(x) = P(X = x)$ | **PDF** $f(x)$ |
| $P(X = x)$ | Can be positive | Always **zero** |
| Total | $\sum_x p(x) = 1$ | $\int f(x)\,dx = 1$ |
| Examples | Number of clicks, defects, heads | Height, temperature, response time |

The surprising row is $P(X = x) = 0$ for continuous variables. There are infinitely many
possible heights, so the probability of being *exactly* 175.000000… cm is zero. For
continuous variables only **intervals** have probability: $P(174 < X < 176)$.

---
## 2.2 Probability Mass Function (PMF)

For a discrete random variable, the PMF lists the probability of each value:

$$p(x) = P(X = x), \qquad p(x) \ge 0, \qquad \sum_x p(x) = 1$$

In [ ]:
# PMF of X = number of heads in 3 flips, derived by counting outcomes
counts = pd.Series([X[o] for o in S]).value_counts().sort_index()
pmf = counts / len(S)

pmf_df = pd.DataFrame({"x": pmf.index, "P(X = x)": pmf.values})
print(pmf_df.to_string(index=False))
print(f"\nSum of PMF = {pmf.sum():.4f}   (must be exactly 1)")

plt.stem(pmf.index, pmf.values)
plt.xlabel("x = number of heads")
plt.ylabel("P(X = x)")
plt.title("PMF of the number of heads in 3 coin flips")
plt.xticks([0, 1, 2, 3])
plt.show()

---
## 2.3 Cumulative Distribution Function (CDF)

The CDF accumulates probability from the left:

$$F(x) = P(X \le x)$$

It always starts at 0, ends at 1, and never decreases. The CDF works for **both** discrete
and continuous variables, which is why theory often prefers it. For discrete variables it
is a staircase; for continuous variables it is a smooth curve.

Practical uses: percentiles, p-values, and quantile-based feature engineering all come
from the CDF.

In [ ]:
cdf = pmf.cumsum()
print(pd.DataFrame({"x": cdf.index, "P(X <= x)": cdf.values.round(4)}).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].stem(pmf.index, pmf.values)
ax[0].set_title("PMF: P(X = x)")
ax[0].set_xlabel("x"); ax[0].set_ylabel("probability")

ax[1].step(np.append(-1, cdf.index), np.append(0, cdf.values), where="post", lw=2)
ax[1].set_title("CDF: P(X <= x)")
ax[1].set_xlabel("x"); ax[1].set_ylabel("cumulative probability")
ax[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

---
## 2.4 Expectation — the long-run average

The **expected value** (or mean) of a discrete random variable is the probability-weighted
average of its values:

$$E[X] = \mu = \sum_x x \cdot p(x)$$

For continuous variables the sum becomes an integral: $E[X] = \int x f(x)\,dx$.

Two things students often get wrong:

- $E[X]$ need **not** be a possible value. The expected number of heads in 3 flips is 1.5,
  which you can never observe.
- $E[X]$ is a property of the *distribution*, not of a sample. The sample mean
  $\bar{x}$ is an *estimate* of it.

In [ ]:
x_vals = pmf.index.to_numpy()
p_vals = pmf.to_numpy()

E_X = np.sum(x_vals * p_vals)
print("Expectation by definition:")
for x, p in zip(x_vals, p_vals):
    print(f"   {x} * {p:.4f} = {x * p:.4f}")
print(f"   E[X] = {E_X:.4f}")

# Empirical check: simulate 200,000 sets of 3 flips
sim = rng.integers(0, 2, size=(200_000, 3)).sum(axis=1)
print(f"\nSample mean of 200,000 simulations = {sim.mean():.4f}")

### A decision-making example

Expectation is how you compare gambles. Should a company run a marketing campaign?

| Outcome | Probability | Profit |
|---|---|---|
| Big success | 0.15 | +₹800,000 |
| Modest success | 0.45 | +₹150,000 |
| Flop | 0.40 | −₹200,000 |

In [ ]:
campaign = pd.DataFrame({
    "outcome":     ["big success", "modest success", "flop"],
    "probability": [0.15, 0.45, 0.40],
    "profit":      [800_000, 150_000, -200_000],
})
campaign["weighted"] = campaign.probability * campaign.profit

print(campaign.to_string(index=False))
print(f"\nE[profit] = {campaign.weighted.sum():,.0f}")
print("Positive expected value -> worth running IF the company can absorb the downside")
print("many times over. A single high-variance bet is a different question entirely.")

---
## 2.5 Variance and standard deviation

Expectation tells you the centre. **Variance** tells you the spread — the average squared
distance from the mean:

$$\operatorname{Var}(X) = \sigma^2 = E\big[(X - \mu)^2\big] = \sum_x (x - \mu)^2 p(x)$$

A far more convenient computational form (prove it by expanding the square):

$$\operatorname{Var}(X) = E[X^2] - (E[X])^2$$

**Standard deviation** $\sigma = \sqrt{\operatorname{Var}(X)}$ is preferred for reporting
because it is in the same units as $X$ (rupees, not rupees-squared).

In [ ]:
var_def      = np.sum((x_vals - E_X) ** 2 * p_vals)
E_X2         = np.sum(x_vals ** 2 * p_vals)
var_shortcut = E_X2 - E_X ** 2

print(f"Var(X) by definition   = {var_def:.4f}")
print(f"Var(X) = E[X^2]-E[X]^2 = {var_shortcut:.4f}")
print(f"SD(X)                  = {np.sqrt(var_def):.4f}")
print(f"\nSimulated variance     = {sim.var():.4f}")

### Sample variance: why divide by $n-1$?

When you estimate variance from data you have to use the *sample* mean $\bar{x}$, which is
itself pulled toward the data. That makes deviations from $\bar{x}$ slightly too small, so
dividing by $n$ **underestimates** the true variance. Dividing by $n-1$ (Bessel's
correction) fixes the bias.

- `numpy` defaults to `ddof=0` (divide by $n$) — the *population* formula
- `pandas` defaults to `ddof=1` (divide by $n-1$) — the *sample* formula

This mismatch bites people constantly. Let's demonstrate the bias.

In [ ]:
true_var = 25.0        # population variance of N(50, 5^2)
n, reps = 5, 20_000

samples = rng.normal(50, 5, size=(reps, n))
biased   = samples.var(axis=1, ddof=0).mean()     # divide by n
unbiased = samples.var(axis=1, ddof=1).mean()     # divide by n-1

print(f"True population variance      : {true_var:.3f}")
print(f"Average of var with ddof=0 (n)  : {biased:.3f}   <- too small")
print(f"Average of var with ddof=1 (n-1): {unbiased:.3f}   <- unbiased")

arr = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print(f"\nnumpy  .var() default (ddof=0): {np.var(arr):.4f}")
print(f"pandas .var() default (ddof=1): {pd.Series(arr).var():.4f}")

---
## 2.6 Properties of expectation and variance

These identities save enormous amounts of work. Let $a, b$ be constants.

**Expectation is linear — always, no conditions:**

$$E[aX + b] = a\,E[X] + b \qquad\qquad E[X + Y] = E[X] + E[Y]$$

The second one holds **even if $X$ and $Y$ are dependent**. That is remarkable and is the
trick behind many elegant proofs.

**Variance is not linear:**

$$\operatorname{Var}(aX + b) = a^2 \operatorname{Var}(X)$$

$$\operatorname{Var}(X + Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) + 2\operatorname{Cov}(X, Y)$$

The last term vanishes **only if $X$ and $Y$ are independent** (or merely uncorrelated).
Note that adding a constant $b$ shifts the mean but leaves the spread untouched, and that
scaling by $a$ multiplies the variance by $a^2$.

In [ ]:
Xs = rng.normal(loc=10, scale=3, size=200_000)
a, b = 4, 25

print(f"E[X]        = {Xs.mean():8.4f}      Var(X)        = {Xs.var():8.4f}")
print(f"E[aX+b]     = {(a*Xs + b).mean():8.4f}      a*E[X]+b      = {a*Xs.mean() + b:8.4f}")
print(f"Var(aX+b)   = {(a*Xs + b).var():8.4f}      a^2*Var(X)    = {a**2 * Xs.var():8.4f}")

# Independent sum: variances add
Ys = rng.normal(loc=-5, scale=4, size=200_000)
print(f"\nVar(X)+Var(Y)      = {Xs.var() + Ys.var():.4f}")
print(f"Var(X+Y) [indep]   = {(Xs + Ys).var():.4f}")
print(f"Var(X+X) [dep!]    = {(Xs + Xs).var():.4f}   <- equals 4*Var(X), not 2*Var(X)")

### Standardisation: the z-score

Combining the two rules gives the most-used transformation in all of statistics:

$$Z = \frac{X - \mu}{\sigma} \quad\Longrightarrow\quad E[Z] = 0,\ \operatorname{Var}(Z) = 1$$

A z-score answers *"how many standard deviations from the mean is this?"* — which makes
values from different scales comparable. `StandardScaler` in scikit-learn is exactly this
formula.

In [ ]:
exam = pd.DataFrame({
    "student": ["Aisha", "Ben", "Chen", "Diya", "Eli"],
    "maths":   [78, 92, 65, 88, 71],       # class mean 75, sd 12
    "physics": [55, 61, 48, 70, 52],       # class mean 50, sd 6
})

for col in ["maths", "physics"]:
    exam[f"z_{col}"] = (exam[col] - exam[col].mean()) / exam[col].std(ddof=0)

print(exam.round(2).to_string(index=False))
print("\nRaw scores are not comparable across subjects; z-scores are.")

### Shape: skewness and kurtosis

Mean and variance describe location and spread. Two more moments describe *shape*:

- **Skewness** — asymmetry. Positive (right) skew means a long right tail: income,
  house prices, session durations. The mean sits to the right of the median.
- **Kurtosis** — tail heaviness. High excess kurtosis means outliers are more likely than
  a Normal distribution would predict (financial returns are the classic case).

For a symmetric distribution skewness is 0. `scipy.stats.kurtosis` reports **excess**
kurtosis, i.e. 0 for a Normal distribution.

In [ ]:
normal_like = rng.normal(0, 1, 50_000)
right_skew  = rng.exponential(1, 50_000)
heavy_tail  = rng.standard_t(df=3, size=50_000)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, data, name in zip(axes, [normal_like, right_skew, heavy_tail],
                          ["Normal (symmetric)", "Exponential (right-skewed)", "t(3) (heavy tails)"]):
    ax.hist(data, bins=80, density=True, color="steelblue", edgecolor="none")
    ax.axvline(data.mean(),      color="crimson", lw=2, label=f"mean={data.mean():.2f}")
    ax.axvline(np.median(data),  color="darkgreen", lw=2, ls="--", label=f"median={np.median(data):.2f}")
    ax.set_title(f"{name}\nskew={stats.skew(data):.2f}, ex.kurt={stats.kurtosis(data):.2f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 2.7 Continuous random variables and the PDF

For a continuous $X$, the **probability density function** $f(x)$ satisfies:

$$P(a \le X \le b) = \int_a^b f(x)\,dx, \qquad \int_{-\infty}^{\infty} f(x)\,dx = 1$$

$f(x)$ is a **density**, not a probability — it can exceed 1! What must stay below 1 is
the *area*. Probability = area under the curve.

In [ ]:
# Uniform(0, 0.5) has density 2.0 everywhere on [0, 0.5] -- a density above 1 is fine
u = stats.uniform(loc=0, scale=0.5)
xs = np.linspace(-0.2, 0.7, 400)

plt.plot(xs, u.pdf(xs), lw=2, color="steelblue")
plt.fill_between(xs, u.pdf(xs), alpha=0.3, color="steelblue")
plt.title("Uniform(0, 0.5): density = 2.0, but total area = 1.0")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.show()

print(f"Max density value : {u.pdf(0.25):.2f}")
print(f"Total area        : {u.cdf(0.5) - u.cdf(0.0):.2f}")

In [ ]:
# Area = probability, illustrated on a Normal distribution
nd = stats.norm(loc=170, scale=8)          # adult height in cm, say
xs = np.linspace(140, 200, 600)
lo, hi = 165, 180

plt.plot(xs, nd.pdf(xs), lw=2, color="steelblue")
mask = (xs >= lo) & (xs <= hi)
plt.fill_between(xs[mask], nd.pdf(xs[mask]), alpha=0.4, color="steelblue")
plt.title(f"P({lo} < X < {hi}) = shaded area = {nd.cdf(hi) - nd.cdf(lo):.4f}")
plt.xlabel("height (cm)"); plt.ylabel("density")
plt.show()

print(f"P(X = 170) exactly      : {0.0}    (always zero for continuous variables)")
print(f"P(169.9 < X < 170.1)    : {nd.cdf(170.1) - nd.cdf(169.9):.5f}")
print(f"P(X < 160)              : {nd.cdf(160):.4f}")
print(f"90th percentile (ppf)   : {nd.ppf(0.90):.2f} cm")

> **`scipy.stats` cheat sheet.** Every distribution object gives you the same methods:
> `.pdf(x)` / `.pmf(k)` — density or mass ·
> `.cdf(x)` — $P(X \le x)$ ·
> `.sf(x)` — survival, $P(X > x)$ ·
> `.ppf(q)` — inverse CDF (quantile) ·
> `.rvs(size)` — random samples ·
> `.mean()`, `.var()`, `.std()`, `.stats()`.
> Learn this interface once and every distribution in Notebook 3 is free.

---
## 2.8 LOTUS — expectation of a function

Suppose you know the distribution of $X$ and you want $E[g(X)]$ — say $E[X^2]$ or
$E[\text{revenue}(X)]$. You do **not** need the distribution of $g(X)$. The *Law of the
Unconscious Statistician* says:

$$E[g(X)] = \sum_x g(x)\,p(x) \qquad\text{or}\qquad \int g(x) f(x)\,dx$$

And a warning that saves many wrong answers:

$$E[g(X)] \ne g(E[X]) \quad \text{in general}$$

For a convex $g$ (like $x^2$), **Jensen's inequality** guarantees $E[g(X)] \ge g(E[X])$.

In [ ]:
# g(x) = x^2 on our coin-flip variable
g_of_E = E_X ** 2
E_of_g = np.sum(x_vals ** 2 * p_vals)

print(f"g(E[X]) = (E[X])^2 = {g_of_E:.4f}")
print(f"E[g(X)] = E[X^2]   = {E_of_g:.4f}")
print(f"Difference = Var(X) = {E_of_g - g_of_E:.4f}   <- exactly the variance!")

In [ ]:
# A business example of Jensen's inequality.
# Demand D ~ Poisson(mean 20). We stock 20 units. Sales = min(D, 20).
D = rng.poisson(20, 200_000)
sales = np.minimum(D, 20)

print(f"E[D]              = {D.mean():.3f}")
print(f"min(E[D], 20)     = {min(D.mean(), 20):.3f}   <- the naive (wrong) forecast")
print(f"E[min(D, 20)]     = {sales.mean():.3f}   <- the truth: you lose sales on high-demand days")
print("\nPlanning with 'average demand' systematically overstates achievable sales.")

---
## 2.9 Joint, marginal and conditional distributions

Real data has many columns. A **joint distribution** $p(x, y) = P(X = x, Y = y)$ describes
two variables together.

- **Marginal:** sum out the other variable — $p_X(x) = \sum_y p(x,y)$ (literally the row
  totals in the margin of the table, hence the name)
- **Conditional:** $p_{Y|X}(y \mid x) = \dfrac{p(x,y)}{p_X(x)}$
- **Independent** iff $p(x,y) = p_X(x)\,p_Y(y)$ for *every* pair

In [ ]:
# X = number of devices owned, Y = subscribed to premium (0/1)
joint = pd.DataFrame(
    [[0.12, 0.03],
     [0.28, 0.17],
     [0.15, 0.25]],
    index=pd.Index([1, 2, 3], name="X = devices"),
    columns=pd.Index([0, 1], name="Y = premium"),
)
print("Joint PMF p(x, y):")
print(joint, "\n")
print(f"Sums to {joint.values.sum():.2f}\n")

marg_X = joint.sum(axis=1)
marg_Y = joint.sum(axis=0)
print("Marginal of X:\n", marg_X.to_string(), "\n")
print("Marginal of Y:\n", marg_Y.to_string())

In [ ]:
# Conditional distribution of Y given each X
cond = joint.div(marg_X, axis=0)
print("P(Y = y | X = x):")
print(cond.round(4), "\n")
print("Premium rate rises with device count -> X and Y are dependent.\n")

# Independence check: compare joint against the product of marginals
product_table = pd.DataFrame(np.outer(marg_X, marg_Y), index=joint.index, columns=joint.columns)
print("If independent, the joint would be:")
print(product_table.round(4))
print(f"\nMax absolute discrepancy: {(joint - product_table).abs().values.max():.4f}  -> NOT independent")

In [ ]:
# E[X], E[Y] and E[XY] from the joint table -> a first look at covariance (Notebook 5)
x_grid, y_grid = np.meshgrid(joint.index.to_numpy(), joint.columns.to_numpy(), indexing="ij")
P = joint.to_numpy()

EX  = (x_grid * P).sum()
EY  = (y_grid * P).sum()
EXY = (x_grid * y_grid * P).sum()

print(f"E[X]  = {EX:.4f}")
print(f"E[Y]  = {EY:.4f}")
print(f"E[XY] = {EXY:.4f}")
print(f"Cov(X, Y) = E[XY] - E[X]E[Y] = {EXY - EX*EY:.4f}  (positive -> they move together)")

---
## 2.10 From a random variable to a dataset

Every dataset column is a **sample** from some underlying random variable. The connection:

| Population (theory) | Sample (data) |
|---|---|
| $\mu = E[X]$ | $\bar{x} = \frac{1}{n}\sum x_i$ |
| $\sigma^2 = \operatorname{Var}(X)$ | $s^2 = \frac{1}{n-1}\sum (x_i - \bar{x})^2$ |
| PMF / PDF | histogram |
| CDF | empirical CDF |

The whole discipline of **inference** (Notebooks 4, 6, 7, 8) is about going from the right
column to the left one.

In [ ]:
sample = rng.normal(loc=170, scale=8, size=500)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

xs = np.linspace(140, 200, 400)
ax[0].hist(sample, bins=30, density=True, alpha=0.6, color="steelblue",
           edgecolor="white", label="sample histogram")
ax[0].plot(xs, stats.norm(170, 8).pdf(xs), lw=2, color="crimson", label="true PDF")
ax[0].set_title("Histogram estimates the PDF"); ax[0].legend()

ax[1].step(np.sort(sample), np.arange(1, 501) / 500, where="post",
           color="steelblue", label="empirical CDF")
ax[1].plot(xs, stats.norm(170, 8).cdf(xs), lw=2, color="crimson", ls="--", label="true CDF")
ax[1].set_title("Empirical CDF estimates the CDF"); ax[1].legend()

plt.tight_layout(); plt.show()

print(f"True mu = 170.00   sample mean = {sample.mean():.2f}")
print(f"True sd =   8.00   sample sd   = {sample.std(ddof=1):.2f}")

---
## Exercises

**Exercise 1.** A game costs ₹50 to play. You roll one fair die and win ₹10 times the
number rolled. Find $E[\text{net profit}]$ and $\operatorname{SD}(\text{net profit})$.
Is the game worth playing?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
faces = np.arange(1, 7)
probs = np.full(6, 1/6)
profit = 10 * faces - 50

E  = (profit * probs).sum()
V  = ((profit - E) ** 2 * probs).sum()

print(pd.DataFrame({"roll": faces, "net profit": profit, "p": probs.round(4)}).to_string(index=False))
print(f"\nE[profit]  = {E:.2f}")
print(f"SD[profit] = {np.sqrt(V):.2f}")
print("Negative expected value -> in the long run you lose 15 rupees per play.")

sim = 10 * rng.integers(1, 7, 200_000) - 50
print(f"\nSimulated mean = {sim.mean():.2f}, sd = {sim.std(ddof=1):.2f}")

**Exercise 2.** Let $X$ be the sum of two fair dice. Build its PMF, compute $E[X]$ and
$\operatorname{Var}(X)$ from the definition, and verify that
$E[X] = 2 \cdot E[\text{one die}]$ and $\operatorname{Var}(X) = 2 \cdot \operatorname{Var}(\text{one die})$.
Why does the variance add here?

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from itertools import product as iproduct

outcomes = [a + b for a, b in iproduct(range(1, 7), repeat=2)]
s = pd.Series(outcomes).value_counts(normalize=True).sort_index()

xv, pv = s.index.to_numpy(), s.to_numpy()
EX = (xv * pv).sum()
VX = ((xv - EX) ** 2 * pv).sum()

one_E = np.mean(faces)
one_V = np.mean((faces - one_E) ** 2)

print(f"E[sum]   = {EX:.4f}   2 * E[one die]   = {2*one_E:.4f}")
print(f"Var[sum] = {VX:.4f}   2 * Var[one die] = {2*one_V:.4f}")
print("\nVariance adds because the two dice are INDEPENDENT, so Cov = 0.")

**Exercise 3.** Website response time is modelled as Exponential with mean 200 ms.
Using `scipy.stats.expon(scale=200)`, find:
(a) $P(X < 100)$  (b) $P(X > 500)$  (c) the 95th percentile (the "p95 latency" your SRE
team cares about)  (d) confirm the mean and sd by simulation.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
exp = stats.expon(scale=200)

print(f"(a) P(X < 100)      = {exp.cdf(100):.4f}")
print(f"(b) P(X > 500)      = {exp.sf(500):.4f}")
print(f"(c) 95th percentile = {exp.ppf(0.95):.1f} ms")

draws = exp.rvs(size=200_000, random_state=1)
print(f"(d) theory  mean={exp.mean():.1f}  sd={exp.std():.1f}")
print(f"    simulated mean={draws.mean():.1f}  sd={draws.std(ddof=1):.1f}")
print("\nNote mean == sd for the exponential, and the p95 is 3x the mean:")
print("that long right tail is why average latency hides user pain.")

**Exercise 4 (challenge).** You buy a stock. Each day it goes up 10% with probability 0.5
or down 10% with probability 0.5, independently. After 100 days, is your expected wealth
above or below your starting capital? Is your *typical* (median) outcome the same?
Simulate 50,000 investors starting with ₹100.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
days, investors = 100, 50_000
moves = np.where(rng.random((investors, days)) < 0.5, 1.10, 0.90)
final = 100 * moves.prod(axis=1)

print(f"Theoretical E[wealth] = 100 * (0.5*1.1 + 0.5*0.9)^100 = {100 * (1.0)**days:.2f}")
print(f"Simulated  mean       = {final.mean():.2f}")
print(f"Simulated  MEDIAN     = {np.median(final):.2f}")
print(f"Fraction who lost money = {(final < 100).mean():.3f}")

plt.hist(np.log10(final), bins=80, color="steelblue", edgecolor="none")
plt.axvline(2, color="crimson", ls="--", lw=2, label="starting capital (100)")
plt.xlabel("log10(final wealth)"); plt.ylabel("number of investors")
plt.title("Mean is flat, but most investors lose money")
plt.legend(); plt.show()

print()
print("The mean is dragged up by a few huge winners (E is linear, growth is multiplicative).")
print("Each up-down pair gives 1.1 * 0.9 = 0.99, so the typical path decays.")
print("This is why compounding returns are summarised with the GEOMETRIC mean.")

---
## Summary

| Concept | Formula |
|---|---|
| PMF (discrete) | $p(x) = P(X = x)$, $\sum p(x) = 1$ |
| PDF (continuous) | $P(a<X<b) = \int_a^b f(x)dx$ |
| CDF | $F(x) = P(X \le x)$ |
| Expectation | $E[X] = \sum x\,p(x)$ |
| Variance | $\operatorname{Var}(X) = E[X^2] - (E[X])^2$ |
| Linearity | $E[aX+b] = aE[X]+b$; $E[X+Y]=E[X]+E[Y]$ always |
| Variance scaling | $\operatorname{Var}(aX+b) = a^2\operatorname{Var}(X)$ |
| Sum of independents | $\operatorname{Var}(X+Y) = \operatorname{Var}(X)+\operatorname{Var}(Y)$ |
| Standardisation | $Z = (X-\mu)/\sigma$ |
| LOTUS | $E[g(X)] = \sum g(x)p(x)$, and $E[g(X)] \ne g(E[X])$ |

**Next up:** [Notebook 3 — Probability Distributions](3.%20Probability%20Distributions.ipynb),
where we meet the named distributions that describe most real-world randomness.